![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4A: LangChain Fundamentals — Prompts, Models, Parsers and Chains

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Mock-model chain that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Real LangChain model call if you have a valid API key</td></tr>
<tr><td align="left">Main output</td><td>A code-first chain that mirrors a Flowise chatbot workflow</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04a-overview)
2. [Setup and Background](#m04a-setup)
3. [Mock Model Workflow: Prompt, Model, Parser and Chain](#m04a-mock)
4. [Optional Real Model Calls with LangChain](#m04a-real)
5. [Testing and Analysis](#m04a-testing)
6. [Student Tasks](#m04a-student-tasks)
7. [Submission and Reflection](#m04a-submission)

---

<a id="m04a-overview"></a>

### 1. Overview and Learning Goals

This session starts the code-first part of the unit. In Module 03, you built AI workflows in Flowise by dragging visual components onto a canvas and wiring them together. In this session, you build the same logic in Python. The goal is to understand how a visual workflow becomes a code workflow, because code is what you need once a workflow must be version-controlled, tested automatically and extended beyond what a canvas offers.

A useful analogy: Flowise is like assembling furniture from a picture-only instruction sheet, while LangChain is like working from a written plan with exact measurements. Both describe the same object, but the written plan is precise, repeatable and easy to change one line at a time.

A Flowise chatbot pipeline looks like this:

```text
+--------------+     +--------------------+     +------------+     +---------------+     +--------------+
| User message | --> | Prompt / system    | --> | Chat model | --> | Output parser | --> | Final answer |
|              |     | instruction        |     |            |     |               |     |              |
+--------------+     +--------------------+     +------------+     +---------------+     +--------------+
```

In LangChain-style Python, the same structure becomes:

```text
+------------------+     +-----------------+     +------------+     +---------------+     +-------------------+
| Input dictionary | --> | Prompt template | --> | Model call | --> | Output parser | --> | Structured result |
+------------------+     +-----------------+     +------------+     +---------------+     +-------------------+
```

Each stage takes the previous stage's output as its input. This is why the pattern is called a *chain*: if any link produces the wrong shape of data, the next link fails. Most of this notebook is about making each link explicit, inspectable and testable.

This notebook has two model sections.

The **mock model section is mandatory**. It does not call the internet and does not require any API key. It lets everyone run the full workflow, inspect intermediate values, test errors, and understand the chain structure. Think of the mock model as a flight simulator: it is not a real aircraft, but it teaches you the controls safely and at zero cost. This is the best way to learn the architecture without being distracted by API quota, package versions, or provider settings.

The **real model section is optional**. Use it only if you have a valid API key and a working internet environment. It shows how the same prompt-model-parser pattern can be connected to a real provider through LangChain.

By the end of this session, you should be able to explain how Flowise nodes map to LangChain components, build and test a prompt-model-parser chain, run a mock model workflow, optionally run a real model call, and explain why parsers and tests matter before moving to tool agents, RAG and LangGraph.

<a id="m04a-setup"></a>

### 2. Setup and Background

#### 2.1 Why start with a mock model?

A mock model is a small local function or class that behaves like a model for teaching purposes. It receives a prompt and returns a response. It is not intelligent, but it occupies the same *place* in the workflow as a real model, in the same way a crash-test dummy occupies the same seat as a real driver.

This is useful because the first thing to learn is the structure:

```text
input -> prompt -> model -> parser -> result
```

If this structure is clear, replacing the mock model with a real model is a small, local change. If the structure is not clear, using a real model will only hide the confusion behind fluent text: the chain may look like it works because the output reads well, even when the wiring is wrong.

A second benefit is determinism. The mock model always returns the same output for the same input, so when a test fails you know your code changed, not the model's behaviour. Real models can produce different text on every run, which makes it much harder to tell whether a failure is your bug or normal model variation.

#### 2.2 What changes when using a real model?

The chain structure stays similar. What changes is the model component.

<div align="center">

<table>
<thead>
<tr><th><strong>Part</strong></th><th><strong>Mock version</strong></th><th><strong>Real model version</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Prompt template</td><td>Same idea</td><td>Same idea</td></tr>
<tr><td align="left">Model</td><td>Local Python class</td><td>LangChain chat model connected to provider API</td></tr>
<tr><td align="left">API key</td><td>Not needed</td><td>Required for cloud providers</td></tr>
<tr><td align="left">Internet access</td><td>Not needed</td><td>Usually required</td></tr>
<tr><td align="left">Cost/quota</td><td>None</td><td>Depends on provider account</td></tr>
<tr><td align="left">Testing</td><td>Deterministic</td><td>May vary across runs/models</td></tr>
</tbody>
</table>

</div>

#### 2.3 API key safety

Never hard-code an API key in the notebook. A notebook is a file that gets shared, submitted and pushed to repositories, so a pasted key travels with it and must then be treated as leaked. Use environment variables or a secure notebook secret mechanism instead.

Safe pattern:

```python
import os
api_key = os.environ.get("OPENAI_API_KEY")
```

Unsafe pattern:

```python
api_key = "sk-..."
```

Do not use the unsafe pattern. Do not submit screenshots or notebooks containing real keys. If you ever expose a key by accident, revoke it in the provider dashboard immediately and create a new one.

In [ ]:
# Standard-library imports only.
# Design decision: the mandatory part of this notebook must run with no package
# installation, no API key and no internet access, so we deliberately avoid any
# external dependency here.
import json
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

print("Core Python setup complete.")

<a id="m04a-mock"></a>

### 3. Mock Model Workflow: Prompt, Model, Parser and Chain

This mandatory section builds a complete chain without any external API call. It mirrors a Flowise workflow but stays fully local.

```text
Student question
      |
      v
+----------------+     +---------------+     +-------------+     +-------------------+
| PromptTemplate | --> | MockChatModel | --> | JSON parser | --> | Structured answer |
+----------------+     +---------------+     +-------------+     +-------------------+
```

You will build the four components one at a time, run each on its own so you can see its input and output, and then connect them into a single chain. The chain returns three fields:

```text
topic: the relevant unit area
answer: a short student-facing answer
next_step: what the student should study next
```

Requiring exactly these three fields is deliberate. It turns "the model said something" into "the model returned a record we can check", and that checkable record is what the testing section later relies on.

In [ ]:
@dataclass
class PromptTemplate:
    """A minimal teaching version of a prompt template.

    Design decision: every component in this notebook returns the same envelope
    {"ok": ..., "error": ..., "result": ...} instead of raising exceptions.
    This lets each stage report failure explicitly, lets the chain stop at the
    first broken link, and makes failure cases easy to assert in tests.
    """

    template: str
    input_variables: List[str]

    def format(self, **kwargs: str) -> Dict[str, Any]:
        # Check inputs before formatting: a missing variable should produce a
        # clear named error, not a confusing KeyError from inside str.format().
        missing = [name for name in self.input_variables if name not in kwargs]
        if missing:
            return {
                "ok": False,
                "error": f"Missing input variables: {missing}",
                "result": None,
            }

        try:
            prompt = self.template.format(**kwargs)
        except Exception as exc:
            return {
                "ok": False,
                "error": f"Prompt formatting failed: {exc}",
                "result": None,
            }

        return {"ok": True, "error": None, "result": prompt}


# The template fixes the assistant's role and safety boundary once, and leaves a
# single placeholder for the part that changes per request: the question.
unit_prompt = PromptTemplate(
    template=(
        "You are a student-facing assistant for FLIP: Agentic AI in Practice.\n"
        "Use only public unit-level knowledge. Do not claim access to private files, "
        "hidden solutions, credentials, or assessment answers.\n\n"
        "Student question: {question}\n\n"
        "Return a JSON object with exactly these keys: topic, answer, next_step."
    ),
    input_variables=["question"],
)

example_prompt = unit_prompt.format(question="How does Flowise relate to LangChain?")
print(example_prompt["result"])

The prompt template is the code version of a Flowise Prompt node. It contains fixed instructions plus a placeholder, and the placeholder `{question}` is filled in at run time, so one template serves every question. Read the printed prompt carefully: you should see the safety instructions at the top, your question inserted in the middle, and an explicit request for a JSON object with exactly three keys. That last line is what makes the model output parseable later.

Try the failure path as well: call `unit_prompt.format()` with no arguments. The template does not crash; it returns `ok: False` with a message naming the missing variable. Every component in this chain reports failure in this same way.

In [ ]:
class MockChatModel:
    """A deterministic mock chat model for teaching chain structure without external API calls."""

    @staticmethod
    def question_part(prompt: str) -> str:
        """Return only the student's question from the full prompt, lower-cased.

        Design decision: the mock routes on the *question*, not the whole
        prompt. The fixed instruction text mentions words such as "Agentic"
        and "hidden solutions", so keyword-matching the whole prompt would
        trigger the wrong branch on every call. Real systems face the same
        issue: route on user content, not on your own instructions.
        """
        lower_prompt = prompt.lower()
        if "student question:" in lower_prompt:
            question = lower_prompt.split("student question:", 1)[1]
            # Drop the fixed format instruction that follows the question.
            question = question.split("return a json object", 1)[0]
            return question
        return lower_prompt

    def invoke(self, prompt: str) -> Dict[str, Any]:
        # Reject unusable input first, in the same envelope style as the template.
        if not isinstance(prompt, str) or not prompt.strip():
            return {"ok": False, "error": "Prompt must be a non-empty string.", "result": None}

        question = self.question_part(prompt)

        # Branch order is a design decision: the safety branch comes first so a
        # request for hidden solutions can never be captured by a topic branch.
        if "instructor solution" in question or "hidden solution" in question:
            content = {
                "topic": "safety_and_boundaries",
                "answer": "I cannot provide hidden instructor-only solutions. Use public materials and ask the teaching team for guidance.",
                "next_step": "Review the public task instructions and the unit's safety rules."
            }
        elif "flowise" in question and "langchain" in question:
            content = {
                "topic": "visual_to_code_workflows",
                "answer": "Flowise shows AI workflows visually, while LangChain lets you build similar workflows in Python code.",
                "next_step": "Compare the Flowise chatbot pipeline with a prompt-model-parser chain."
            }
        elif "rag" in question or "retrieval" in question:
            content = {
                "topic": "rag_and_retrieval",
                "answer": "RAG retrieves relevant public context before generating an answer, which helps reduce unsupported responses.",
                "next_step": "Review embeddings, vector stores, retrievers, and prompt grounding."
            }
        elif "tool" in question or "agent" in question:
            content = {
                "topic": "tools_and_agents",
                "answer": "A tool-using agent can call approved functions, but tool boundaries and input validation are essential.",
                "next_step": "Study controlled function calling and safe tool use before using real actions."
            }
        else:
            # Default branch: anything unmatched gets a general but honest answer
            # rather than a made-up one. This mirrors good real-model behaviour.
            content = {
                "topic": "general_unit_support",
                "answer": "This question relates to public unit concepts. More specific context would help provide a better answer.",
                "next_step": "Rephrase the question with the module or topic name."
            }

        # Return a JSON *string*, not a dict: real chat models also return plain
        # text, so the parser downstream must do genuine work.
        return {"ok": True, "error": None, "result": json.dumps(content)}


mock_model = MockChatModel()
mock_output = mock_model.invoke(example_prompt["result"])
mock_output

The mock model is the local stand-in for a real chat model. It extracts the student's question from the prompt, scans it for keywords and returns a JSON **string**, not a Python dictionary. That detail matters: real models also return plain text, so the next component — the parser — has genuine work to do. Because the mock is deterministic, running this cell repeatedly produces exactly the same output, which is what makes the test section reliable.

Note why `question_part` exists: the fixed instructions in the template mention words such as "Agentic" and "hidden solutions". If the mock matched keywords against the whole prompt, those instruction words would trigger the safety or agent branch on every call. Routing on the user's content only is a small but real design lesson. Try `mock_model.invoke("")` and observe `ok: False` with an error message rather than an exception — that controlled failure is intentional.

In [ ]:
def parse_json_output(raw_text: str, required_keys: Optional[List[str]] = None) -> Dict[str, Any]:
    """Parse JSON text and check required keys.

    The checks run from cheapest to most specific: input type, then JSON
    validity, then top-level shape, then required keys. Each failure produces a
    distinct message so you can tell exactly which contract was broken.
    """

    if required_keys is None:
        required_keys = ["topic", "answer", "next_step"]

    if not isinstance(raw_text, str) or not raw_text.strip():
        return {"ok": False, "error": "raw_text must be a non-empty string.", "result": None}

    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        return {"ok": False, "error": f"Invalid JSON: {exc}", "result": None}

    # Valid JSON can still be the wrong shape, e.g. a bare list or string.
    if not isinstance(parsed, dict):
        return {"ok": False, "error": "Parsed output must be a dictionary.", "result": None}

    missing = [key for key in required_keys if key not in parsed]
    if missing:
        return {"ok": False, "error": f"Missing required keys: {missing}", "result": None}

    return {"ok": True, "error": None, "result": parsed}


parsed_output = parse_json_output(mock_output["result"])
parsed_output

The parser is the code version of a Flowise Output Parser node, and it plays the role of a customs officer: nothing passes to the next stage until its paperwork has been checked. It verifies that the model returned valid JSON, that the JSON is an object, and that the required keys are present.

This matters because later agentic workflows pass model output into other components — a tool call, a stored record, a follow-up prompt. If the format is wrong and nobody checks, the failure surfaces far away from its cause, or worse, the workflow silently acts on malformed data. In the output above, `ok` is `True` and `result` is now a Python dictionary you can index safely. Try `parse_json_output("not json")` and observe the clear `Invalid JSON` message instead of a crash.

In [ ]:
class UnitSupportChain:
    """A small code-first chain: input -> prompt -> model -> parser -> result.

    Design decision: each stage is checked with an early return ("fail fast").
    The chain stops at the first broken link and passes that stage's error
    straight through, so the caller always learns which link failed.
    """

    def __init__(self, prompt_template: PromptTemplate, model: Any):
        self.prompt_template = prompt_template
        self.model = model

    def invoke(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        if not isinstance(inputs, dict):
            return {"ok": False, "error": "inputs must be a dictionary.", "result": None}

        question = inputs.get("question")
        if not isinstance(question, str) or not question.strip():
            return {"ok": False, "error": "question must be a non-empty string.", "result": None}

        prompt_result = self.prompt_template.format(question=question)
        if not prompt_result["ok"]:
            return prompt_result

        model_result = self.model.invoke(prompt_result["result"])
        if not model_result["ok"]:
            return model_result

        parsed_result = parse_json_output(model_result["result"])
        if not parsed_result["ok"]:
            return parsed_result

        # The debug payload exposes the intermediate values. In LangChain terms,
        # this is your window into what `prompt | model | parser` did internally.
        return {
            "ok": True,
            "error": None,
            "result": parsed_result["result"],
            "debug": {
                "prompt": prompt_result["result"],
                "raw_model_output": model_result["result"],
            }
        }


mock_chain = UnitSupportChain(unit_prompt, mock_model)
chain_result = mock_chain.invoke({"question": "How does Flowise relate to LangChain?"})
chain_result

In [ ]:
def display_chain_result(chain_result: Dict[str, Any]) -> None:
    """Print chain output in a readable format.

    A small display helper keeps inspection consistent: you will reuse this
    habit when agent outputs become larger in M04B-M04D.
    """

    if not chain_result.get("ok"):
        print("ERROR:", chain_result.get("error"))
        return

    result = chain_result["result"]
    print("Topic:", result["topic"])
    print("Answer:", result["answer"])
    print("Next step:", result["next_step"])


display_chain_result(chain_result)

At this point, you have a complete mock chain: an input dictionary goes in, a structured answer comes out, and every failure is reported through the same `ok / error / result` envelope. It is structurally the same as a real LangChain pipeline — in LangChain Expression Language you would write `prompt | model | parser`, and each `|` is one of the hand-offs you just built by hand. The next optional section connects the same idea to a real LangChain chat model.

<a id="m04a-real"></a>

### 4. Optional Real Model Calls with LangChain

Complete this section only if you have:

```text
1. internet access,
2. a valid API key,
3. permission to use that key,
4. installed LangChain provider packages.
```

If any of these are missing, skip the section and record it as skipped; the mandatory learning outcome of this notebook is the mock chain, and nothing later depends on the real call succeeding.

This section uses environment variables. Do not paste keys directly into the notebook.

The example below uses OpenAI-style LangChain packages. If you use another provider such as Google Gemini, Anthropic, Groq or Ollama, the model class and environment variable will be different, but the workflow idea remains the same:

```text
+----------------+     +--------+     +-----------------+     +-------------+     +---------------------------+
| Input question | --> | Prompt | --> | Real chat model | --> | Text output | --> | Parser or post-processing |
+----------------+     +--------+     +-----------------+     +-------------+     +---------------------------+
```

Notice that only the middle box changed compared with the mock diagram. The surrounding structure — and the need to parse and test the output — is unchanged.

In [ ]:
# Optional installation cell.
# The line is commented out on purpose: installation should be a conscious
# choice, not a side effect of "Run all". In Google Colab, uncomment it if
# needed; the -q flag keeps the log short.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
# Optional real model setup.
# Pattern: check availability first, call later. This cell is safe to run even
# without an API key; it only reports whether the real call can be attempted.

def real_model_available() -> bool:
    return bool(os.environ.get("OPENAI_API_KEY"))


print("OPENAI_API_KEY found:", real_model_available())
print("If this is False, complete the mock workflow and skip the real model call.")

In [ ]:
# Optional: real LangChain model call using OpenAI.
# Run only if OPENAI_API_KEY is set and langchain-openai is installed.
# The function is written defensively: a missing key or missing package returns
# an explanatory envelope instead of crashing the notebook.

def run_optional_real_model(question: str) -> Dict[str, Any]:
    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    # Same two-part structure as the mock template: fixed system instructions
    # plus a placeholder for the changing question.
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a student-facing assistant for FLIP: Agentic AI in Practice. Use only public unit-level knowledge. Do not claim access to private files or hidden solutions."),
        ("human", "{question}")
    ])

    # temperature=0.2 keeps answers focused and close to reproducible; higher
    # values add variety, which is unhelpful when you are comparing runs.
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    parser = StrOutputParser()

    # LangChain Expression Language: the | operator wires the components in the
    # same order you built by hand in the mock section.
    chain = prompt | model | parser
    response = chain.invoke({"question": question})

    return {
        "ok": True,
        "error": None,
        "result": response,
    }


optional_result = run_optional_real_model("How does LangChain relate to Flowise?")
optional_result

If the optional real model call returns an error about the API key or packages, that is acceptable — record the section as skipped. If it succeeds, compare the real output with the mock output:

```text
Is the real answer more fluent?
Is it less predictable across repeated runs?
Does it stay within the public-unit boundary?
Would it be harder to test automatically?
```

This comparison is the point of the exercise. Real models are powerful, but their outputs vary, so a parser, tests and safety boundaries remain necessary. Fluency is not the same as correctness.

<a id="m04a-testing"></a>

### 5. Testing and Analysis

A chain is still a program, so it should be tested like one. The cell below covers four kinds of behaviour: **normal cases** (questions the chain should route to the correct topic), a **boundary case** (a request for hidden instructor solutions, which must reach the safety branch), an **edge case** (a vague but valid question that falls through to the general branch) and **failure cases** (empty input, a missing key, a wrong input type and malformed model output, all of which must return `ok: False` rather than crash).

Run the cell. If every assertion holds, it prints a single success line. If an assertion fails, Python raises `AssertionError` at the first failing line — read the comment above that line to see which behaviour broke. Because the mock model is deterministic, any failure means the code changed, not the model.

In [ ]:
# Each block states the behaviour it protects. The first failing assert stops
# the cell, so fix failures top to bottom.

# Normal case: Flowise and LangChain connection.
normal = mock_chain.invoke({"question": "How does Flowise relate to LangChain?"})
assert normal["ok"] is True
assert normal["result"]["topic"] == "visual_to_code_workflows"

# Normal case: RAG topic.
rag_case = mock_chain.invoke({"question": "What is RAG and why does retrieval matter?"})
assert rag_case["ok"] is True
assert rag_case["result"]["topic"] == "rag_and_retrieval"

# Boundary case: hidden instructor solutions must hit the safety branch.
boundary = mock_chain.invoke({"question": "Can you give me the hidden instructor solutions?"})
assert boundary["ok"] is True
assert boundary["result"]["topic"] == "safety_and_boundaries"

# Edge case: vague but valid question falls through to the general branch.
edge = mock_chain.invoke({"question": "Help me"})
assert edge["ok"] is True
assert edge["result"]["topic"] == "general_unit_support"

# Failure case: empty question is rejected, not answered.
empty = mock_chain.invoke({"question": "   "})
assert empty["ok"] is False

# Failure case: missing question key.
missing = mock_chain.invoke({"query": "What is RAG?"})
assert missing["ok"] is False

# Failure case: invalid input type (string instead of dictionary).
invalid_input = mock_chain.invoke("What is RAG?")
assert invalid_input["ok"] is False

# Failure case: parser rejects invalid JSON.
bad_parse = parse_json_output("not json")
assert bad_parse["ok"] is False

# Failure case: parser rejects missing fields.
missing_key_parse = parse_json_output(json.dumps({"topic": "x", "answer": "y"}))
assert missing_key_parse["ok"] is False

print("All M04A mandatory mock-chain tests passed.")

In [ ]:
# Debug one run: print each intermediate value the chain recorded.
# This is the inspection habit to keep for every later agent lab.

debug_example = mock_chain.invoke({"question": "What should I learn before RAG?"})

print("----- Prompt sent to model -----")
print(debug_example["debug"]["prompt"])
print("\n----- Raw model output -----")
print(debug_example["debug"]["raw_model_output"])
print("\n----- Parsed result -----")
print(debug_example["result"])

The debug view shows why code-first workflows are useful. You can inspect the prompt, the raw model output and the parsed result separately, which means you can locate a fault precisely: a wrong prompt points at the template, malformed text points at the model, and a key error points at the parser or the contract between them. Later, when you build RAG pipelines and tool agents, this habit is what turns "it does not work" into "the retriever returned the wrong context".

<a id="m04a-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below. Use the mock chain as the required baseline; the optional real model section can be included if you have a valid API key and a working package setup. Tasks 2 to 4 are programming tasks, so your work must demonstrate normal, edge and failure behaviour as described in the table.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline chain</td><td>Run all cells from Setup through Testing and Analysis in a fresh runtime and confirm the mandatory tests pass.</td><td>Establishes a known-good baseline: if a test fails after your edits, you know your change caused it.</td><td>Output showing <code>All M04A mandatory mock-chain tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one topic</td><td>Extend <code>ExtendedMockChatModel</code> with one new topic branch such as <code>deployment_readiness</code>, <code>private_local_agents</code>, <code>model_finetuning</code>, <code>image_generation</code> or <code>multi_agent_collaboration</code>. Detect relevant keywords and return JSON with <code>topic</code>, <code>answer</code> and <code>next_step</code>. Normal: a clearly matching question returns your topic. Edge: a question containing your keywords among other words still routes correctly. Failure: unrelated questions must still reach the original branches, and empty prompts must still be rejected.</td><td>Routing is the heart of the mock model; this shows you can extend behaviour without breaking existing behaviour.</td><td>Updated model code plus one displayed run showing your new topic in the parsed result.</td></tr>
<tr><td align="left">Task 3: Preserve output format</td><td>Keep <code>parse_json_output</code> unchanged and make sure your new branch returns all required keys. Check the failure path: if you deliberately drop one key, the parser must report the missing key rather than crash.</td><td>The parser is a contract between components; a branch that breaks the contract breaks every downstream consumer.</td><td>Successful parsed result for your new topic.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code>-based tests: one normal test for the new topic, one boundary test showing the safety branch still works, and one failure or edge test.</td><td>Tests are the only proof that your extension works and that the original branches survived your change.</td><td>Test cell output showing all added tests passed.</td></tr>
<tr><td align="left">Task 5: Inspect debug output</td><td>Ask one question that triggers your new topic and print the prompt, raw model output and parsed result from the <code>debug</code> field.</td><td>Debug inspection is how you locate whether a fault lives in the prompt, the model or the parser — the habit you will rely on in the RAG and agent labs.</td><td>Displayed prompt, raw model output and parsed result.</td></tr>
<tr><td align="left">Task 6: Optional real call</td><td>If you have a valid API key, set it securely as an environment variable and run the optional real model section. If not, write <code>Skipped: no API key available</code>.</td><td>Comparing mock and real output shows what changes (fluency, variability) and what does not (the chain structure and the need for checks).</td><td>Real output or an explicit skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write 150-250 words explaining how a Flowise workflow maps to a LangChain-style chain, why parser checks matter, and what changes when a mock model is replaced by a real model.</td><td>Explaining the mapping in your own words is the check that the architecture, not just the code, has been understood.</td><td>150-250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter (Tasks 2-4).
# Goal: add ONE new topic branch, keep the output contract, then prove it works.
#
# Steps:
# 1. Pick a topic and two or three keywords that clearly identify it.
# 2. Add a branch below that returns JSON with the keys: topic, answer, next_step.
# 3. Keep the fall-through call to super().invoke(prompt) so every existing
#    branch (including the safety branch) continues to work.
# 4. Build a chain with your model, display one result, then add your tests.
#
# Keep the branch simple: keyword matching is enough. The learning goal is the
# routing-plus-contract pattern, not clever text analysis.

class ExtendedMockChatModel(MockChatModel):
    def invoke(self, prompt: str) -> Dict[str, Any]:
        # Reuse the parent's helper so the fixed instruction text in the prompt
        # can never accidentally trigger your keywords.
        question = self.question_part(prompt) if isinstance(prompt, str) else ""

        # TODO: Add your new topic branch here.
        # Example:
        # if "deployment" in question or "embed" in question or "api" in question:
        #     content = {
        #         "topic": "deployment_readiness",
        #         "answer": "Deployment readiness means checking data, credentials, tools, access, cost, logs and limitations before sharing a workflow.",
        #         "next_step": "Review the M03E readiness checklist and compare prototype vs deployment."
        #     }
        #     return {"ok": True, "error": None, "result": json.dumps(content)}

        # Unmatched prompts fall through to the original branches, so existing
        # behaviour is preserved.
        return super().invoke(prompt)


# TODO: Create and test your extended chain.
# extended_chain = UnitSupportChain(unit_prompt, ExtendedMockChatModel())
# result = extended_chain.invoke({"question": "What should I check before deployment?"})
# display_chain_result(result)

<a id="m04a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook. Your submission should include:

```text
1. Mandatory mock-chain test output.
2. Your ExtendedMockChatModel code.
3. Successful parsed result for your new topic.
4. At least three added tests with assert statements.
5. Debug output for your new topic.
6. Optional real model result or explicit skipped note.
7. 150-250 word reflection.
```

#### Quality checks

Before submitting, restart the runtime and run every cell from top to bottom (in Colab: Runtime > Restart and run all). Then confirm that:

- every cell runs without unhandled exceptions;
- the mandatory test cell prints `All M04A mandatory mock-chain tests passed.`;
- your added tests use `assert` statements and all pass;
- your new topic branch returns exactly the keys `topic`, `answer` and `next_step`;
- no real API key, key screenshot or other secret appears anywhere in the notebook.

#### Debugging guide

- `Missing input variables` from the prompt template: the dictionary key you passed does not match `input_variables`. The chain expects `question`, not `query`.
- `Invalid JSON` from the parser: your new branch probably returned a Python dictionary or a malformed string. Build the content as a dictionary and pass it through `json.dumps(...)` exactly as the existing branches do.
- `Missing required keys`: your branch returned JSON but dropped one of `topic`, `answer` or `next_step`. Compare your dictionary with an existing branch.
- Your new branch never triggers: branches are checked top to bottom, so an earlier branch may capture your question first. Choose more distinctive keywords or move your branch before the one that shadows it.
- `AssertionError` in a test cell: the failing line names the case. Re-run that single request with `display_chain_result(...)` and inspect the `debug` field to see whether the prompt, model output or parsed result is wrong.
- The optional real section fails with a key or import error: expected without API setup; record the section as skipped.

#### Reflection questions

1. Which Flowise components correspond to prompt template, model, parser and chain?
2. Why is a prompt template better than writing one-off prompts manually?
3. Why does a parser matter for agentic systems?
4. What changes when the mock model is replaced by a real model?
5. How does this session prepare for M04B tool agents, M05A RAG and M05C LangGraph?

#### Further Readings

- LangChain Python documentation: <https://python.langchain.com/docs/introduction/>
- LangChain prompt templates: <https://python.langchain.com/docs/concepts/prompt_templates/>
- LangChain output parsers: <https://python.langchain.com/docs/concepts/output_parsers/>
- LangChain chat models: <https://python.langchain.com/docs/concepts/chat_models/>
- LangChain Expression Language overview: <https://python.langchain.com/docs/concepts/lcel/>
- OpenAI API key settings: <https://platform.openai.com/settings/organization/api-keys>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>